In [ ]:
from libraries import *
from parameters import *


In [ ]:
os.getcwd()
os.chdir(projectDir)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
adata = sc.read("./Data/adataALL.h5ad")

In [ ]:
adata = adata[(adata.obs["guide_num"] == 1),]

In [ ]:
adata.obs

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad

def inverse_cluster_sample(
    adata: ad.AnnData,
    cluster_key: str = "leiden",
    n_total: int = 20_000,
    alpha: float = 0.5,
    random_state: int = 0,
) -> ad.AnnData:
    """
    Sample cells such that expected sampling weight per cluster is proportional to 1/(cluster_size^alpha).
    alpha=1.0 => strictly inverse proportional.
    """

    rng = np.random.default_rng(random_state)

    clust = adata.obs[cluster_key].astype(str)
    counts = clust.value_counts().sort_index()  # cluster -> size
    clusters = counts.index.to_numpy()
    n = counts.to_numpy()

    if n_total >= adata.n_obs:
        return adata.copy()

    # weights per cluster: w_k ∝ 1 / n_k^alpha
    w = 1.0 / (n.astype(float) ** alpha)
    w = w / w.sum()

    # initial desired allocations
    target = np.floor(w * n_total).astype(int)

    # ensure at least 1 from each cluster (optional; comment out if you don't want this)
    target = np.maximum(target, 1)

    # can't take more than exists
    target = np.minimum(target, n)

    # adjust to sum exactly n_total while respecting caps
    # (greedy distribute remaining or remove extras based on available capacity)
    diff = int(n_total - target.sum())

    # capacity remaining per cluster
    cap = n - target

    if diff > 0:
        # add remaining samples to clusters with remaining capacity, proportional to weights
        while diff > 0:
            eligible = np.where(cap > 0)[0]
            if eligible.size == 0:
                break  # can't reach n_total without replacement
            probs = w[eligible]
            probs = probs / probs.sum()
            # add in batches
            add = min(diff, eligible.size * 10)
            chosen = rng.choice(eligible, size=add, replace=True, p=probs)
            # apply adds but respect caps
            for idx in chosen:
                if cap[idx] > 0 and diff > 0:
                    target[idx] += 1
                    cap[idx] -= 1
                    diff -= 1

    elif diff < 0:
        # remove extras from clusters with target>0, preferentially from larger targets
        diff = -diff
        while diff > 0:
            eligible = np.where(target > 0)[0]
            probs = target[eligible].astype(float)
            probs = probs / probs.sum()
            remove = min(diff, eligible.size * 10)
            chosen = rng.choice(eligible, size=remove, replace=True, p=probs)
            for idx in chosen:
                if target[idx] > 0 and diff > 0:
                    target[idx] -= 1
                    cap[idx] += 1
                    diff -= 1

    # Now actually sample indices within each cluster
    sampled_idx = []
    for c, k in enumerate(clusters):
        take = int(target[c])
        if take <= 0:
            continue
        idx_k = np.where(clust.values == k)[0]
        sampled_idx.append(rng.choice(idx_k, size=take, replace=False))

    sampled_idx = np.concatenate(sampled_idx)
    rng.shuffle(sampled_idx)

    # If we still couldn't reach n_total (rare; happens if many clusters are tiny and got capped),
    # top up by sampling remaining cells uniformly from the rest.
    if sampled_idx.size < n_total:
        remaining = np.setdiff1d(np.arange(adata.n_obs), sampled_idx, assume_unique=False)
        need = n_total - sampled_idx.size
        extra = rng.choice(remaining, size=need, replace=False)
        sampled_idx = np.concatenate([sampled_idx, extra])
        rng.shuffle(sampled_idx)

    return adata[sampled_idx].copy()


# usage
adata_20k = inverse_cluster_sample(adata, cluster_key="leiden", n_total=20000, alpha=0.1, random_state=1)

In [ ]:
sc.pl.umap(adata, color="leiden", size=0.5,
           color_map="coolwarm", vmax=2.0)


In [ ]:
sc.pl.umap(adata_20k, color="leiden", size=0.3,
           color_map="coolwarm", vmax=2.0)


In [ ]:
adata_20k.obs['leiden'].value_counts()

In [ ]:
adata_20k.write("./Data/adata_20K.h5ad")

In [ ]:
adata = sc.read("./Data/adata_20K.h5ad")

In [ ]:
import anndata as ad
X = adata.layers["counts"]

# cNMF requires non-negative counts
if sp.issparse(X):
    if (X.data < 0).any():
        raise ValueError("adata.layers['counts'] contains negative values; cNMF requires non-negative counts.")
else:
    X_arr = np.asarray(X)
    if (X_arr < 0).any():
        raise ValueError("adata.layers['counts'] contains negative values; cNMF requires non-negative counts.")

# Minimal AnnData for cNMF: counts in .X, keep obs_names/var_names
# Avoid heavy copies; this is usually enough for cNMF I/O
adata_counts = ad.AnnData(
    X=X,
    obs=pd.DataFrame(index=adata.obs_names),
    var=pd.DataFrame(index=adata.var_names),
)

adata_counts.write_h5ad("./Data/adata_counts_20K.h5ad")


In [ ]:
from cnmf import cNMF

cnmf_obj = cNMF(output_dir=str("./CNMF_out"), name="deneme")

In [ ]:
cnmf_obj.prepare(
    counts_fn=str("./Data/adata_counts_20K.h5ad"),
    components=np.array(list([7]), dtype=int),
    n_iter=int(100),
    seed=int(1),
)

In [ ]:
cnmf_obj.factorize(worker_i=0, total_workers=1)


In [ ]:
cnmf_obj.combine()

In [ ]:
cnmf_obj.k_selection_plot()

In [ ]:
cnmf_obj.consensus(k=7, density_threshold=0.02)


In [ ]:
usage, spectra_scores, spectra_tpm, top_genes = cnmf_obj.load_results(
        K=int(7),
        density_threshold=float(0.02))

In [ ]:
top_genes.to_csv("Programs_K7.csv")

In [ ]:
# Label outputs
cells = adata.obs_names.to_list()
genes = adata.var_names.to_list()
programs = [f"CNMF_k{k_final}_p{i+1}" for i in range(int(k_final))]

usage_df = pd.DataFrame(usage, index=cells, columns=programs)
spectra_tpm_df = pd.DataFrame(spectra_tpm, index=programs, columns=genes)
spectra_scores_df = pd.DataFrame(spectra_scores, index=programs, columns=genes)


spectra_tpm_df.to_csv("Programs_tmp_k10.csv")
spectra_scores_df.to_csv("ProgramScores_k10.csv")


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import multiprocessing as mp
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad


def _set_thread_env(blas_threads: int) -> None:
    # Limit BLAS/OpenMP threads per worker (critical for performance)
    os.environ["OMP_NUM_THREADS"] = str(blas_threads)
    os.environ["OPENBLAS_NUM_THREADS"] = str(blas_threads)
    os.environ["MKL_NUM_THREADS"] = str(blas_threads)
    os.environ["VECLIB_MAXIMUM_THREADS"] = str(blas_threads)
    os.environ["NUMEXPR_NUM_THREADS"] = str(blas_threads)


def _factorize_one(args: tuple[str, str, int, int, int]) -> int:
    out_dir, name, worker_i, total_workers, blas_threads = args
    _set_thread_env(blas_threads)
    cnmf_obj = cNMF(output_dir=out_dir, name=name)
    cnmf_obj.factorize(worker_i=worker_i, total_workers=total_workers)
    return worker_i


def run_cnmf_parallel(
    adata: ad.AnnData,
    out_dir: str | Path,
    name: str,
    k_list=(8, 10, 12, 14),
    k_final: int = 10,
    n_iter: int = 100,
    seed: int = 14,
    density_threshold: float = 0.01,
    n_workers: int = 32,
    blas_threads_per_worker: int = 1,
    programs_csv_prefix: str | Path | None = None,
) -> dict:
    """
    Faster cNMF on a single large node by parallelizing factorize() across workers.

    Recommended settings for 128 cores:
      - n_workers: 32 or 64
      - blas_threads_per_worker: 1
    """

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if "counts" not in adata.layers:
        raise KeyError(f"adata.layers['counts'] not found. Available layers: {list(adata.layers.keys())}")

    X = adata.layers["counts"]

    # cNMF requires non-negative counts
    if sp.issparse(X):
        if (X.data < 0).any():
            raise ValueError("adata.layers['counts'] contains negative values; cNMF requires non-negative counts.")
    else:
        X_arr = np.asarray(X)
        if (X_arr < 0).any():
            raise ValueError("adata.layers['counts'] contains negative values; cNMF requires non-negative counts.")

    # Minimal AnnData for cNMF: counts in .X, keep obs_names/var_names
    # Avoid heavy copies; this is usually enough for cNMF I/O
    adata_counts = ad.AnnData(
        X=X,
        obs=pd.DataFrame(index=adata.obs_names),
        var=pd.DataFrame(index=adata.var_names),
    )

    counts_fn = out_dir / f"{name}.counts.h5ad"
    adata_counts.write_h5ad(counts_fn)

    # Prepare (single process)
    _set_thread_env(blas_threads_per_worker)  # also limit threads in main proc
    cnmf_obj = cNMF(output_dir=str(out_dir), name=name)
    cnmf_obj.prepare(
        counts_fn=str(counts_fn),
        components=np.array(list(k_list), dtype=int),
        n_iter=int(n_iter),
        seed=int(seed),
    )

    # Factorize in parallel (multiple processes)
    n_workers = int(n_workers)
    tasks = [(str(out_dir), name, i, n_workers, int(blas_threads_per_worker)) for i in range(n_workers)]

    # Use spawn for safety on some systems; fork is faster on Linux but can inherit bad state
    ctx = mp.get_context("spawn")
    with ctx.Pool(processes=n_workers) as pool:
        for wid in pool.imap_unordered(_factorize_one, tasks, chunksize=1):
            pass  # could print progress if you want

    # Combine + consensus (single process)
    cnmf_obj = cNMF(output_dir=str(out_dir), name=name)
    cnmf_obj.combine()
    cnmf_obj.k_selection_plot()
    cnmf_obj.consensus(k=int(k_final), density_threshold=float(density_threshold))

    usage, spectra_scores, spectra_tpm, top_genes = cnmf_obj.load_results(
        K=int(k_final),
        density_threshold=float(density_threshold),
    )

    # Label outputs
    cells = adata.obs_names.to_list()
    genes = adata.var_names.to_list()
    programs = [f"{name}_k{k_final}_p{i+1}" for i in range(int(k_final))]

    usage_df = pd.DataFrame(usage, index=cells, columns=programs)
    spectra_tpm_df = pd.DataFrame(spectra_tpm, index=programs, columns=genes)
    spectra_scores_df = pd.DataFrame(spectra_scores, index=programs, columns=genes)

    # Save programs
    if programs_csv_prefix is None:
        programs_csv_prefix = out_dir / f"{name}_k{k_final}_programs"
    else:
        programs_csv_prefix = Path(programs_csv_prefix)

    programs_tpm_csv = programs_csv_prefix.with_suffix("")  # strip suffix if any
    programs_tpm_csv = Path(str(programs_tpm_csv) + "_tpm.csv")
    programs_scores_csv = Path(str(programs_csv_prefix.with_suffix("")) + "_scores.csv")

    spectra_tpm_df.to_csv(programs_tpm_csv)
    spectra_scores_df.to_csv(programs_scores_csv)

    return {
        "usage_df": usage_df,
        "spectra_tpm_df": spectra_tpm_df,
        "spectra_scores_df": spectra_scores_df,
        "top_genes": top_genes,
        "paths": {
            "counts_fn": str(counts_fn),
            "out_dir": str(out_dir),
            "programs_tpm_csv": str(programs_tpm_csv),
            "programs_scores_csv": str(programs_scores_csv),
        },
    }


# Example (good starting point for 128 cores):
# - 64 workers × 1 BLAS thread each = uses ~64 cores efficiently, leaves headroom
results = run_cnmf_parallel(
    adata,
    out_dir="./cnmf_out",
    name="my_run",
    k_list=(8, 10, 12),
    k_final=10,
    n_iter=100,
    n_workers=64,
    blas_threads_per_worker=1,
)
print(results["paths"])